In [1]:
# ══════════════════════════════════════════════════════════════════
# ML Leakage & Pipeline Assignment — Complete Solution
# ══════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

# Generate synthetic dataset (used across all tasks)
X, y = make_classification(n_samples=1000, n_features=10, random_state=42)

# ══════════════════════════════════════════════════════════════════
# TASK 1 — Reproduce and Identify Leakage (FLAWED approach)
# ══════════════════════════════════════════════════════════════════

# ❌ WRONG: Scaling on the ENTIRE dataset before splitting
# This leaks test set statistics into training — the scaler learns
# the mean and std of the test set too, which it should never see.

scaler_leaky = StandardScaler()
X_scaled_leaky = scaler_leaky.fit_transform(X)  # ← leakage happens here

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_scaled_leaky, y, test_size=0.2, random_state=42
)

model_leaky = LogisticRegression(random_state=42)
model_leaky.fit(X_train_l, y_train_l)

train_acc_leaky = model_leaky.score(X_train_l, y_train_l)
test_acc_leaky  = model_leaky.score(X_test_l,  y_test_l)

print("=" * 55)
print("TASK 1 — Flawed Approach (Data Leakage Present)")
print("=" * 55)
print(f"Train Accuracy : {train_acc_leaky:.4f}")
print(f"Test  Accuracy : {test_acc_leaky:.4f}")
print("""
❌ What is wrong:
   StandardScaler was fit on the ENTIRE dataset (X) before
   the train/test split. This means the scaler has already
   seen the test samples when computing mean and std deviation.
   The model indirectly benefits from test-set information
   during training — this is data leakage.
   In production the model will perform worse than reported
   here because real unseen data won't be pre-scaled this way.
""")

# ══════════════════════════════════════════════════════════════════
# TASK 2 — Fix with Pipeline + Cross-Validation (CORRECT approach)
# ══════════════════════════════════════════════════════════════════

# ✅ CORRECT: Split FIRST, then let the Pipeline handle scaling
# The scaler is fit ONLY on training folds — never sees test data.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),       # fit only on train fold
    ('model',  LogisticRegression(random_state=42))
])

# 5-Fold Cross-Validation on training set only
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy')

# Final evaluation on held-out test set
pipeline.fit(X_train, y_train)
test_acc_pipeline = pipeline.score(X_test, y_test)

print("=" * 55)
print("TASK 2 — Corrected Pipeline Approach")
print("=" * 55)
print(f"CV Fold Accuracies : {np.round(cv_scores, 4)}")
print(f"Mean CV Accuracy   : {cv_scores.mean():.4f}")
print(f"Std  CV Accuracy   : {cv_scores.std():.4f}")
print(f"Test Accuracy      : {test_acc_pipeline:.4f}")
print("""
✅ Why this is correct:
   The Pipeline ensures StandardScaler is fit only on the
   training fold in each CV iteration. The test fold (and
   final holdout set) is only transformed — never used to
   fit the scaler. This gives an honest, leakage-free
   estimate of how the model will perform on new data.
""")

# ══════════════════════════════════════════════════════════════════
# TASK 3 — Decision Tree Depth Experiment
# ══════════════════════════════════════════════════════════════════

# Using the same X_train / X_test split from Task 2 (no scaling
# needed — Decision Trees are scale-invariant).

print("=" * 55)
print("TASK 3 — Decision Tree Depth Experiment")
print("=" * 55)

depths       = [1, 5, 20]
results      = []

for depth in depths:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    train_acc = dt.score(X_train, y_train)
    test_acc  = dt.score(X_test,  y_test)
    results.append({
        'max_depth'       : depth,
        'Train Accuracy'  : round(train_acc, 4),
        'Test Accuracy'   : round(test_acc,  4),
    })

results_df = pd.DataFrame(results).set_index('max_depth')
print(results_df.to_string())

print("""
Explanation:
  max_depth=1  → Underfitting. The tree is too shallow to learn
                 meaningful patterns. Both train and test accuracy
                 are low — high bias, low variance.

  max_depth=5  → Best balance. Train and test accuracy are close
                 to each other, showing the model generalises well
                 without memorising the training data.

  max_depth=20 → Overfitting. The tree memorises the training set
                 (train accuracy ≈ 1.0) but test accuracy drops
                 because it has learned noise, not signal.

  ✅ max_depth=5 best balances bias and variance.
""")

TASK 1 — Flawed Approach (Data Leakage Present)
Train Accuracy : 0.8700
Test  Accuracy : 0.8300

❌ What is wrong:
   StandardScaler was fit on the ENTIRE dataset (X) before
   the train/test split. This means the scaler has already
   seen the test samples when computing mean and std deviation.
   The model indirectly benefits from test-set information
   during training — this is data leakage.
   In production the model will perform worse than reported
   here because real unseen data won't be pre-scaled this way.

TASK 2 — Corrected Pipeline Approach
CV Fold Accuracies : [0.8812 0.8938 0.875  0.8438 0.8562]
Mean CV Accuracy   : 0.8700
Std  CV Accuracy   : 0.0179
Test Accuracy      : 0.8300

✅ Why this is correct:
   The Pipeline ensures StandardScaler is fit only on the
   training fold in each CV iteration. The test fold (and
   final holdout set) is only transformed — never used to
   fit the scaler. This gives an honest, leakage-free
   estimate of how the model will perform on ne